<a href="https://colab.research.google.com/github/arnav307/Concept-and-technology-of-AI/blob/main/week8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Step -1- Building a Custom Decision Tree with Information Gain:

In [1]:
import numpy as np

class CustomDecisionTree: #this function will be used in iris dataset part
    def __init__(self, max_depth=None):
        """
        Initializes the decision tree with the specified maximum depth.

        Parameters:
        max_depth (int, optional): The maximum depth of the tree. If None, the tree is expanded until all
        leaves are pure or contain fewer than the minimum samples required to split.
        """
        self.max_depth = max_depth
        self.tree = None

    def fit(self, X, y):
        """
        Trains the decision tree model using the provided training data.

        Parameters:
        X (array-like): Feature matrix (n_samples, n_features) for training the model.
        y (array-like): Target labels (n_samples,) for training the model.
        """
        self.tree = self._build_tree(X, y)

    def _build_tree(self, X, y, depth=0):
        """
        Recursively builds the decision tree by splitting the data based on the best feature and threshold.

        Parameters:
        X (array-like): Feature matrix (n_samples, n_features) for splitting.
        y (array-like): Target labels (n_samples,) for splitting.
        depth (int, optional): Current depth of the tree during recursion.

        Returns:
        dict: A dictionary representing the structure of the tree, containing the best feature index,
        threshold, and recursive tree nodes.
        """
        num_samples, num_features = X.shape

        #handling the edge case = empty node
        if num_samples == 0:
            return {"class": 0}

        unique_classes = np.unique(y)

        # Stopping conditions: pure node or reached max depth
        if len(unique_classes) == 1:
            return {"class": unique_classes[0]}
        if self.max_depth is not None and depth >= self.max_depth:
            return {"class": np.bincount(y).argmax()}

        #to find the best split based on Information Gain (using Entropy)
        best_info_gain = -float("inf")
        best_split = None

        for feature_idx in range(num_features):
            thresholds = np.unique(X[:, feature_idx])
            for threshold in thresholds:
                left_mask = X[:, feature_idx] <= threshold
                right_mask = ~left_mask

                left_y = y[left_mask]
                right_y = y[right_mask]

                # Skip invalid splits
                if len(left_y) == 0 or len(right_y) == 0:
                    continue

                info_gain = self._information_gain(y, left_y, right_y)

                if info_gain > best_info_gain:
                    best_info_gain = info_gain
                    best_split = {
                        "feature_idx": feature_idx,
                        "threshold": threshold,
                        "left_mask": left_mask,
                        "right_mask": right_mask,
                    }

        if best_split is None:
            return {"class": np.bincount(y).argmax()}

        # Recursively build the left and right subtrees
        left_tree = self._build_tree(X[best_split["left_mask"]], y[best_split["left_mask"]], depth + 1)
        right_tree = self._build_tree(X[best_split["right_mask"]], y[best_split["right_mask"]], depth + 1)

        return {
            "feature_idx": best_split["feature_idx"],
            "threshold": best_split["threshold"],
            "left_tree": left_tree,
            "right_tree": right_tree,
        }

    def _information_gain(self, parent, left, right):
        """
        Computes the Information Gain between the parent node and the left/right child nodes.

        Parameters:
        parent (array-like): The labels of the parent node.
        left (array-like): The labels of the left child node.
        right (array-like): The labels of the right child node.

        Returns:
        float: The Information Gain of the split.
        """
        parent_entropy = self._entropy(parent)
        left_entropy = self._entropy(left)
        right_entropy = self._entropy(right)

        # Information Gain = Entropy(parent) - (weighted average of left and right entropies)
        weighted_avg_entropy = (len(left) / len(parent)) * left_entropy + (len(right) / len(parent)) * right_entropy
        return parent_entropy - weighted_avg_entropy

    def _entropy(self, y):
        """
        Computes the entropy of a set of labels.

        Parameters:
        y (array-like): The labels for which entropy is calculated.

        Returns:
        float: The entropy of the labels.
        """
        class_probs = np.bincount(y) / len(y)
        return -np.sum(class_probs * np.log2(class_probs + 1e-9))  # Added small epsilon to avoid log(0)

    def predict(self, X):
        """
        Predicts the target labels for the given test data based on the trained decision tree.

        Parameters:
        X (array-like): Feature matrix (n_samples, n_features) for prediction.

        Returns:
        list: A list of predicted target labels (n_samples,).
        """
        return [self._predict_single(x, self.tree) for x in X]

    def _predict_single(self, x, tree):
        """
        Recursively predicts the target label for a single sample by traversing the tree.

        Parameters:
        x (array-like): A single feature vector for prediction.
        tree (dict): The current subtree or node to evaluate.

        Returns:
        int: The predicted class label for the sample.
        """
        if "class" in tree:
            return tree["class"]

        feature_val = x[tree["feature_idx"]]
        if feature_val <= tree["threshold"]:
            return self._predict_single(x, tree["left_tree"])
        else:
            return self._predict_single(x, tree["right_tree"])

In [2]:
# Step 2 = Load and Split Iris (imports only for Iris split)

In [3]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

#iris dataset loading
data = load_iris()
X = data.data
y = data.target

#splitting into train and test sets - 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Step 3 = Train and Evaluate a Custom Decision Tree (imports only for accuracy metric):

In [4]:
from sklearn.metrics import accuracy_score

#to train the custom decision tree
custom_tree = CustomDecisionTree(max_depth=3) #function call from step 1
custom_tree.fit(X_train, y_train)

#predict on the test set
y_pred_custom = custom_tree.predict(X_test)

#calculating accuracy
accuracy_custom = accuracy_score(y_test, y_pred_custom)
print(f"Custom Decision Tree Accuracy: {accuracy_custom:.4f}")

Custom Decision Tree Accuracy: 0.9667


In [ ]:
# Step 4 = Train and Evaluate Scikit Learn Decision Tree (imports only for sklearn tree):

In [5]:
#training and evaluating a Scikit learn decision tree:

from sklearn.tree import DecisionTreeClassifier

#train the Scikit-learn decision tree
sklearn_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
sklearn_tree.fit(X_train, y_train)

#predict on the test set
y_pred_sklearn = sklearn_tree.predict(X_test)

#calculating the accuracy
accuracy_sklearn = accuracy_score(y_test, y_pred_sklearn)
print(f"Scikit-learn Decision Tree Accuracy: {accuracy_sklearn:.4f}")


Scikit-learn Decision Tree Accuracy: 0.9667


In [6]:
print(f"Accuracy Comparison:")
print(f"Custom Decision Tree: {accuracy_custom:.4f}")
print(f"Scikit-learn Decision Tree: {accuracy_sklearn:.4f}")

Accuracy Comparison:
Custom Decision Tree: 0.9667
Scikit-learn Decision Tree: 0.9667


In [ ]:
# 3 Exercise - Ensemble Methods and Hyperparameter Tuning.
# Using the Wine Dataset from scikit-learn

In [7]:
from sklearn.datasets import load_wine
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

#loading the Wine dataset
wine = load_wine()
Xw = wine.data
yw = wine.target

#train-test split
Xw_train, Xw_test, yw_train, yw_test = train_test_split(Xw, yw, test_size=0.2, random_state=42, stratify=yw)

#Implement Classification Models:
#training a decision tree classifier and a random forest classifier using scikit-learn
dt_clf = DecisionTreeClassifier(random_state=42)
rf_clf = RandomForestClassifier(random_state=42)

dt_clf.fit(Xw_train, yw_train)
rf_clf.fit(Xw_train, yw_train)

yw_pred_dt = dt_clf.predict(Xw_test)
yw_pred_rf = rf_clf.predict(Xw_test)

#comparing the models based on their F1 scores
f1_dt = f1_score(yw_test, yw_pred_dt, average="weighted")
f1_rf = f1_score(yw_test, yw_pred_rf, average="weighted")

print(f"Decision Tree (Wine) F1 (weighted): {f1_dt:.4f}")
print(f"Random Forest (Wine) F1 (weighted): {f1_rf:.4f}")


Decision Tree (Wine) F1 (weighted): 0.9450
Random Forest (Wine) F1 (weighted): 1.0000


In [ ]:
# 2. Hyperparameter Tuning:
# • Identify three hyperparameters of the Random Forest Classifier.
# • Perform hyperparameter tuning using GridSearchCV to optimize these parameters.
# • Take hints from the scikit-learn documentation to guide the implementation.

In [8]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 3, 5, 10],
    "min_samples_split": [2, 5, 10],
}

grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring="f1_weighted",
    cv=5,
    n_jobs=-1,
)

grid.fit(Xw_train, yw_train)

best_rf_clf = grid.best_estimator_
yw_pred_best = best_rf_clf.predict(Xw_test)
f1_best = f1_score(yw_test, yw_pred_best, average="weighted")

print("Best GridSearchCV Params (RF Classifier) are:", grid.best_params_)
print(f"Best CV F1 (weighted) is: {grid.best_score_:.4f}")
print(f"Test F1 (weighted) with Best RF is: {f1_best:.4f}")

Best GridSearchCV Params (RF Classifier) are: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 50}
Best CV F1 (weighted) is: 0.9860
Test F1 (weighted) with Best RF is: 1.0000


In [14]:

#Implement Regression Model:

#training a Decision Tree Regressor and a Random Forest Regressor using scikit-learn
#identifying three params for Random Forest Regression and Perform hyperparameter tuning using
#RandomSearchCV to optimize these params

import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

#using California Housing dataset for regression
housing = fetch_california_housing()
Xr = housing.data
yr = housing.target

Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=42)

dt_reg = DecisionTreeRegressor(random_state=42)
rf_reg = RandomForestRegressor(random_state=42)

dt_reg.fit(Xr_train, yr_train)
rf_reg.fit(Xr_train, yr_train)

yr_pred_dt = dt_reg.predict(Xr_test)
yr_pred_rf = rf_reg.predict(Xr_test)

rmse_dt = np.sqrt(mean_squared_error(yr_test, yr_pred_dt))
rmse_rf = np.sqrt(mean_squared_error(yr_test, yr_pred_rf))

r2_dt = r2_score(yr_test, yr_pred_dt)
r2_rf = r2_score(yr_test, yr_pred_rf)

print(f"Decision Tree Regressor RMSE: {rmse_dt:.4f} | R2 (R Sqaure): {r2_dt:.4f}")
print(f"Random Forest Regressor RMSE: {rmse_rf:.4f} | R2 (R Square): {r2_rf:.4f}")

Decision Tree Regressor RMSE: 0.7037 | R2 (R Sqaure): 0.6221
Random Forest Regressor RMSE: 0.5053 | R2 (R Square): 0.8051


In [15]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

param_dist = {
    "n_estimators": [30, 60, 100],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
}

rand_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1, max_samples=0.7),
    param_distributions=param_dist,
    n_iter=6,
    scoring="neg_root_mean_squared_error",
    cv=2,
    random_state=42,
    n_jobs=-1,
)

rand_search.fit(Xr_train, yr_train)

best_rf_reg = rand_search.best_estimator_
yr_pred_best_reg = best_rf_reg.predict(Xr_test)

rmse_best = np.sqrt(mean_squared_error(yr_test, yr_pred_best_reg))
r2_best = r2_score(yr_test, yr_pred_best_reg)

print("Best RandomizedSearchCV Params (RF Regressor) are:", rand_search.best_params_)
print(f"Best CV RMSE (positive) is: {(-rand_search.best_score_):.4f}")
print(f"Test RMSE with Best RF Regressor is: {rmse_best:.4f}")
print(f"Test R2 with Best RF Regressor is: {r2_best:.4f}")

Best RandomizedSearchCV Params (RF Regressor) are: {'n_estimators': 100, 'min_samples_split': 10, 'max_depth': None}
Best CV RMSE (positive) is: 0.5377
Test RMSE with Best RF Regressor is: 0.5127
Test R2 with Best RF Regressor is: 0.7994
